# Libraries

In [ ]:
import os
import csv
import time
import yaml
import shutil
import random
import kagglehub
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt 
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

# Utils

### DOWNLOAD=True if you want to import into Input in Kaggle Notebook and download in Google Colab

CHANGE `DATASET_PATH` TO APPROPRIATE PATH IF ON GOOGLE COLAB

https://www.kaggle.com/datasets/melikechan/cifar100

In [ ]:
DOWNLOAD = False

#### Insert your username below

In [ ]:
username = ""
MODEL_PATH = Path(f"/kaggle/input/models/{username}/wrn/pytorch/default/1")
CONFIG_PATH = Path(f"/kaggle/input/datasets/{username}/wide-resnet/")
DATASET_PATH = "/kaggle/input/datasets/melikechan/cifar100/cifar100"

In [ ]:
def load_yaml(path):
    with open(path, "r") as f:
        cfg = yaml.safe_load(f)
    return cfg

def set_seed(seed: int = 42):
    random.seed(seed)                     # Python random
    np.random.seed(seed)                  # NumPy
    torch.manual_seed(seed)               # CPU
    torch.cuda.manual_seed(seed)          # GPU
    torch.cuda.manual_seed_all(seed)      # All GPUs
    torch.backends.cudnn.deterministic = True  # Deterministic convs
    torch.backends.cudnn.benchmark = False     # Disable auto-tuner for reproducibility
    print(f"Random seed set to {seed}")

def save_training_plots(
    model_name,
    loss_history,
    train_acc_history,
    test_acc_history,
    epoch_times,
    output_dir="outputs/plots"
):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    epochs = np.arange(1, len(loss_history) + 1)

    # Loss plot
    plt.figure()
    plt.plot(epochs, loss_history, label="Train Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Training Loss - {model_name}")
    plt.legend()
    plt.grid(True)
    plt.savefig(output_dir / f"{model_name}_loss.png")
    plt.close()

    # Accuracy plot
    plt.figure()
    plt.plot(epochs, train_acc_history, label="Train Accuracy")
    plt.plot(epochs, test_acc_history, label="Test Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"Train vs Test Accuracy - {model_name}")
    plt.legend()
    plt.grid(True)
    plt.savefig(output_dir / f"{model_name}_accuracy.png")
    plt.close()

    # Time per epoch plot
    plt.figure()
    plt.plot(epochs, epoch_times, label="Time per Epoch (s)")
    plt.xlabel("Epoch")
    plt.ylabel("Seconds")
    plt.title(f"Epoch Time - {model_name}")
    plt.legend()
    plt.grid(True)
    plt.savefig(output_dir / f"{model_name}_epoch_time.png")
    plt.close()

    print(f"\nPlots saved to: {output_dir.resolve()}")

def summarize_checkpoint_times(ckpt_path):
    ckpt = torch.load(MODEL_PATH / ckpt_path, map_location="cpu")
    
    # Check if epoch_times exists
    if "epoch_times" not in ckpt:
        print("Checkpoint does not contain 'epoch_times'.")
        return None

    epoch_times = ckpt["epoch_times"]
    total_time = sum(epoch_times)
    avg_time = total_time / len(epoch_times)

    def format_hms(seconds):
        h = int(seconds // 3600)
        m = int((seconds % 3600) // 60)
        s = int(seconds % 60)
        return f"{h}h {m}m {s}s"

    print(f"Average epoch time: {format_hms(avg_time)}")
    print(f"Total training time: {format_hms(total_time)}")
    
    return avg_time, total_time

In [ ]:
WRN_CFG = load_yaml(CONFIG_PATH / "wrn.yaml")
RESNET_CFG = load_yaml(CONFIG_PATH / "resnet.yaml")
DATA_CFG = load_yaml(CONFIG_PATH / "data.yaml")

# Dataset

In [ ]:
def download_data(data_dir):
    data_dir = Path("/kaggle/working") / data_dir
    data_dir.mkdir(parents=True, exist_ok=True)

    download_path = kagglehub.dataset_download("melikechan/cifar100")

    print("Path to dataset files:", download_path)

    return download_path

In [ ]:
if DOWNLOAD:
    download_data(DATA_CFG['root'])

In [ ]:
def build_transforms(image_size=32, train=True):
    if train:
        return T.Compose([
            T.RandomCrop(image_size, padding=4),
            T.RandomHorizontalFlip(),
            T.ToTensor(),
            T.Normalize(mean=DATA_CFG["mean"], std=DATA_CFG["std"])
        ])
    else:
        return T.Compose([
            T.ToTensor(),
            T.Normalize(mean=DATA_CFG["mean"], std=DATA_CFG["std"])
        ])


class CIFAR100(Dataset):
    def __init__(self, root, split="train", transform=None):
        self.root = Path(root)
        self.transform = transform

        self.data_dir = self.root / split
        self.data_path = []
        self.class_to_idx = {class_dir.name: idx for idx, class_dir in enumerate(self.data_dir.iterdir()) if class_dir.is_dir()}
        self.idx_to_class = {idx: class_name for class_name, idx in self.class_to_idx.items()}

        for class_dir in self.data_dir.iterdir():
            if class_dir.is_dir():
                for img_path in class_dir.iterdir():
                    if img_path.suffix in [".jpg", ".png"]:
                        self.data_path.append((img_path, self.class_to_idx[class_dir.name]))

    def __len__(self):
        return len(self.data_path)
    
    def __getitem__(self, index):
        img_path, label = self.data_path[index]

        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = int(label)
        return image, label

# Model

## WRN

In [ ]:
class wrn_block(nn.Module):
    def __init__(self, in_channels, out_channels, k, identity_downsample=None, stride=1, withDropout=False):
        super(wrn_block, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels*k, kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels*k)
        self.dropout = nn.Dropout(p=0.3)
        self.conv2 = nn.Conv2d(out_channels*k, out_channels*k, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels*k)
        self.relu = nn.ReLU(inplace=True)

        self.identity_downsample = identity_downsample

        self.withDropout = withDropout

    def forward(self, x):
        identity = x

        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)

        if self.withDropout:
            x = self.dropout(x)

        x = self.conv2(x)
        x = self.bn2(x)

        if self.identity_downsample is not None:
            identity = self.identity_downsample(identity)
        
        x += identity
        x = self.relu(x)
        return x
    
class WRN(nn.Module):
    def __init__(self, wrn_block, k, N, image_channels, num_classes, withDropout):
        super(WRN, self).__init__()
        self.in_channels = 16
        self.conv1 = nn.Conv2d(image_channels, 16, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.relu = nn.ReLU(inplace=True)

        # Number of blocks in each layer is determined by N, where N = 6n + 4
        n = (N - 4) // 6

        # WRN Layers
        self.layer1 = self._make_layer(wrn_block, k, n, out_channels=16, stride=1, withDropout=withDropout)
        self.layer2 = self._make_layer(wrn_block, k, n, out_channels=32, stride=2, withDropout=withDropout)
        self.layer3 = self._make_layer(wrn_block, k, n, out_channels=64, stride=2, withDropout=withDropout)

        self.avgpool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(64*k, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)

        x = self.avgpool(x)
        x = x.reshape(x.shape[0], -1)
        x = self.fc(x)

        return x

    def _make_layer(self, wrn_block, k, n, out_channels, stride, withDropout):
        identity_downsample = None
        layers = []

        if stride != 1 or self.in_channels != out_channels * k:
            identity_downsample = nn.Sequential(nn.Conv2d(self.in_channels, out_channels*k, kernel_size=1,
                                                          stride=stride),
                                                nn.BatchNorm2d(out_channels*k))
        
        layers.append(wrn_block(self.in_channels, out_channels, k, identity_downsample, stride, withDropout))
        self.in_channels = out_channels * k

        for i in range(n - 1):
            layers.append(wrn_block(self.in_channels, out_channels, k, withDropout=withDropout))
        
        return nn.Sequential(*layers)
    
def WRN28_10(img_channels=3, num_classes=100, withDropout=False):
    return WRN(wrn_block, 10, 28, img_channels, num_classes, withDropout)

## ResNet

In [ ]:
class resnet_block(nn.Module):
    def __init__(self, in_channels, out_channels, identity_downsample=None, stride=1):
        super(resnet_block, self).__init__()
        self.expansion = 4
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, padding=0)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.conv3 = nn.Conv2d(out_channels, out_channels*self.expansion, kernel_size=1, stride=1, padding=0)
        self.bn3 = nn.BatchNorm2d(out_channels*self.expansion)
        self.relu = nn.ReLU()
        self.identity_downsample = identity_downsample

    def forward(self, x):
        identity = x

        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.conv3(x)
        x = self.bn3(x)

        if self.identity_downsample is not None:
            identity = self.identity_downsample(identity)
        
        x += identity
        x = self.relu(x)
        return x
    
class ResNet(nn.Module):
    def __init__(self, resnet_block, layers, image_channels, num_classes):
        super(ResNet, self).__init__()
        self.in_channels = 64
        self.conv1 = nn.Conv2d(image_channels, 64, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU()

        # ResNet Layers
        self.layer1 = self._make_layer(resnet_block, layers[0], out_channels=64, stride=1)
        self.layer2 = self._make_layer(resnet_block, layers[1], out_channels=128, stride=2)
        self.layer3 = self._make_layer(resnet_block, layers[2], out_channels=256, stride=2)
        self.layer4 = self._make_layer(resnet_block, layers[3], out_channels=512, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(512*4, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = x.reshape(x.shape[0], -1)
        x = self.fc(x)

        return x

    def _make_layer(self, resnet_block, num_residual_blocks, out_channels, stride):
        identity_downsample = None
        layers = []

        if stride != 1 or self.in_channels != out_channels * 4:
            identity_downsample = nn.Sequential(nn.Conv2d(self.in_channels, out_channels*4, kernel_size=1,
                                                          stride=stride),
                                                nn.BatchNorm2d(out_channels*4))
        
        layers.append(resnet_block(self.in_channels, out_channels, identity_downsample, stride))
        self.in_channels = out_channels * 4

        for i in range(num_residual_blocks - 1):
            layers.append(resnet_block(self.in_channels, out_channels))
        
        return nn.Sequential(*layers)
    
def ResNet50(img_channels=3, num_classes=100):
    return ResNet(resnet_block, [3, 4, 6, 3], img_channels, num_classes)
    
def ResNet101(img_channels=3, num_classes=100):
    return ResNet(resnet_block, [3, 4, 23, 3], img_channels, num_classes)
    
def ResNet152(img_channels=3, num_classes=100):
    return ResNet(resnet_block, [3, 8, 36, 3], img_channels, num_classes)

# Train

In [ ]:
def train(model_name):
    # Config
    set_seed(42)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)
    
    # Datasets & loaders
    train_dataset = CIFAR100(
        root=DATASET_PATH, 
        split="train", 
        transform=build_transforms(DATA_CFG["image_size"], train=True))
    test_dataset = CIFAR100(
        root=DATASET_PATH, 
        split="test", 
        transform=build_transforms(DATA_CFG["image_size"], train=False))
    
    train_loader = DataLoader(train_dataset, 
                              batch_size=DATA_CFG["batch_size"], 
                              shuffle=True, 
                              num_workers=DATA_CFG["num_workers"],
                              drop_last=True)
    test_loader = DataLoader(test_dataset, 
                              batch_size=DATA_CFG["batch_size"], 
                              shuffle=False, 
                              num_workers=DATA_CFG["num_workers"],
                              drop_last=True)
    
    # Model, loss, optimizer
    if model_name == "wrn28_10":
        model = WRN28_10(img_channels=3, num_classes=DATA_CFG.get("num_classes", 100), withDropout=True).to(device)
        MODEL_CFG = WRN_CFG
    elif model_name == "resnet50":
        model = ResNet50(img_channels=3, num_classes=DATA_CFG.get("num_classes", 100)).to(device)
        MODEL_CFG = RESNET_CFG
    elif model_name == "resnet101":
        model = ResNet101(img_channels=3, num_classes=DATA_CFG.get("num_classes", 100)).to(device)
        MODEL_CFG = RESNET_CFG
    elif model_name == "resnet152":
        model = ResNet152(img_channels=3, num_classes=DATA_CFG.get("num_classes", 100)).to(device)
        MODEL_CFG = RESNET_CFG
    
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.SGD(
        model.parameters(),
        lr=float(MODEL_CFG.get("lr", 0.001)),
        momentum=float(MODEL_CFG.get("momentum", 0.9)),
        weight_decay=float(MODEL_CFG.get("weight_decay", 1e-4)),
        nesterov=True
    )
    scheduler = optim.lr_scheduler.StepLR(
        optimizer,
        step_size=int(MODEL_CFG.get("step_size", 30)),
        gamma=float(MODEL_CFG.get("gamma", 0.1)),
    )
    
    # Checkpoint
    num_epochs = MODEL_CFG.get("epochs", 50)
    output_dir = Path("/kaggle/working/outputs/checkpoints")
    output_dir.mkdir(parents=True, exist_ok=True)
    
    start_epoch = 1
    best_acc = 0.0
    
    loss_history = []
    train_acc_history = []
    test_acc_history = []
    epoch_times = []

    if MODEL_CFG.get("start_from", None) is not None and not isinstance(MODEL_CFG.get("start_from", None), str):
        ckpt_epoch = int(MODEL_CFG["start_from"])
        ckpt_path = MODEL_PATH / f"{model_name}_epoch_{ckpt_epoch}.pth"

        checkpoint = torch.load(ckpt_path, map_location=device)

        model.load_state_dict(checkpoint["model_state"])
        optimizer.load_state_dict(checkpoint["optimizer_state"])
        scheduler.load_state_dict(checkpoint["scheduler_state"])

        best_acc = checkpoint.get("best_acc", 0.0)

        loss_history = checkpoint.get("loss_history", [])
        train_acc_history = checkpoint.get("train_acc_history", [])
        test_acc_history = checkpoint.get("test_acc_history", [])
        epoch_times = checkpoint.get("epoch_times", [])

        start_epoch = checkpoint["epoch"] + 1

        print(f"Resumed from epoch {start_epoch}")

    # Training loop
    for epoch in range(start_epoch, num_epochs+1):
        start_time = time.time()

        # Training
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        for images, labels in tqdm(train_loader, desc=f"[Train] Epoch {epoch}/{num_epochs}"):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct_train += (preds == labels).sum().item()
            total_train += labels.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)
        train_acc = correct_train / total_train
        loss_history.append(epoch_loss)
        train_acc_history.append(train_acc)

        # Testing
        model.eval()
        correct_test = 0
        total_test = 0
        with torch.no_grad():
            for images, labels in tqdm(test_loader, desc=f"[Test] Epoch {epoch}/{num_epochs}"):
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, preds = torch.max(outputs, 1)
                correct_test += (preds == labels).sum().item()
                total_test += labels.size(0)

        test_acc = correct_test / total_test
        test_acc_history.append(test_acc)

        epoch_time = time.time() - start_time
        epoch_times.append(epoch_time)

        print(f"Epoch {epoch} | Loss: {epoch_loss:.4f} | Train Acc: {train_acc*100:.2f}% | Test Acc: {test_acc*100:.2f}% | Time: {epoch_time:.2f}s")

        # Save plots
        save_training_plots(
            model_name=model_name,
            loss_history=loss_history,
            train_acc_history=train_acc_history,
            test_acc_history=test_acc_history,
            epoch_times=epoch_times,
            output_dir="outputs/plots"
        )

        # Save checkpoint
        ckpt = {
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "best_acc": best_acc,
        
            # histories
            "loss_history": loss_history,
            "train_acc_history": train_acc_history,
            "test_acc_history": test_acc_history,
            "epoch_times": epoch_times,
        }

        ckpt_path = output_dir / f"{model_name}_epoch_{epoch}.pth"
        torch.save(ckpt, ckpt_path)

        if test_acc > best_acc:
            best_acc = test_acc
            best_ckpt_path = output_dir / f"{model_name}_best.pth"
            torch.save(ckpt, best_ckpt_path)
            print(f"Saved best model to {best_ckpt_path}")
        
        scheduler.step()
    
    print("\nTraining Summary")
    print(f"Best Val Accuracy: {best_acc*100:.2f}%")
    print(f"Total time: {sum(epoch_times):.2f} seconds")
    print(f"Avg time/epoch: {np.mean(epoch_times):.2f} seconds")
    print(f"Min epoch time: {np.min(epoch_times):.2f} seconds")
    print(f"Max epoch time: {np.max(epoch_times):.2f} seconds")

if __name__ == "__main__":
    model_name = "wrn28_10"
    train(model_name)

## Inference

In [ ]:
def inference(params_path, topk=(1,5)):
    model_name = "_".join(params_path.split("_")[:2])
    # Setup
    set_seed(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    # Create output directory for plots
    plots_dir = Path(f"outputs/plots/{model_name}")
    plots_dir.mkdir(parents=True, exist_ok=True)

    # Create output directory for metrics
    metric_dir = Path("/kaggle/working/outputs/metrics")
    metric_dir.mkdir(parents=True, exist_ok=True)

    # Data
    test_dataset = CIFAR100(
        root=DATASET_PATH, 
        split="test",  
        transform=build_transforms(DATA_CFG["image_size"], train=False)
    )
    test_loader = DataLoader(
        test_dataset, batch_size=64, shuffle=False, num_workers=1
    )

    idx_to_class = test_dataset.idx_to_class

    # Model, loss, optimizer
    if model_name == "wrn28_10":
        model = WRN28_10(img_channels=3, num_classes=DATA_CFG.get("num_classes", 100), withDropout=True).to(device)
    elif "resnet50" in model_name:
        model_name = params_path.split("_")[0]
        model = ResNet50(img_channels=3, num_classes=DATA_CFG.get("num_classes", 100)).to(device)
    elif "resnet101" in model_name:
        model_name = params_path.split("_")[0]
        model = ResNet101(img_channels=3, num_classes=DATA_CFG.get("num_classes", 100)).to(device)
    elif "resnet152" in model_name:
        model_name = params_path.split("_")[0]
        model = ResNet152(img_channels=3, num_classes=DATA_CFG.get("num_classes", 100)).to(device)
        
    ckpt_path = MODEL_PATH / params_path
    checkpoint = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(checkpoint["model_state"])
    model.eval()

    # Metrics Tracking
    total = 0
    topk_correct = [0] * len(topk)
    confusion_counter = Counter()      # (true, pred)
    per_class_total = Counter()        # true
    per_class_correct = Counter()      # true & correct

    # Inference Loop
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc=f"[Inference]"):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(images)
            probs = torch.softmax(logits, dim=1)

            # Top-k accuracy
            for i, k in enumerate(topk):
                topk_preds = torch.topk(probs, k, dim=1).indices
                topk_correct[i] += (
                    topk_preds == labels.unsqueeze(1)
                ).any(dim=1).sum().item()

            # Top-1 predictions
            preds = torch.argmax(probs, dim=1)

            for t, p in zip(labels.cpu().numpy(), preds.cpu().numpy()):
                per_class_total[t] += 1
                if t == p:
                    per_class_correct[t] += 1
                else:
                    confusion_counter[(t, p)] += 1

            total += labels.size(0)

    # Print accuracy
    print("\nAccuracy:")
    for i, k in enumerate(topk):
        acc = topk_correct[i] / total
        print(f"Top-{k}: {acc:.4f}")

    # Confusion analysis
    most_confused = confusion_counter.most_common(10)

    print("\nTop 10 most confused class pairs (true -> predicted):")
    for (t, p), count in most_confused:
        print(f"{idx_to_class[t]} -> {idx_to_class[p]} : {count}")

    if most_confused:
        # Bar plot for top 10 most confused
        labels_plot = [
            f"{idx_to_class[t]}->{idx_to_class[p]}"
            for (t, p), _ in most_confused
        ]
        counts = [c for _, c in most_confused]

        plt.figure(figsize=(10, 5))
        plt.bar(range(len(counts)), counts)
        plt.xticks(range(len(counts)), labels_plot, rotation=45)
        plt.ylabel("Count")
        plt.title("Top 10 Most Confused Class Pairs")
        plt.tight_layout()

        plot_path = plots_dir / f"{model_name}_most_confused_pairs.png"
        plt.savefig(plot_path)
        plt.close()
        print(f"\nConfusion plot saved to: {plot_path}")

        # Automatic Top-10 Confused Image Grid

        fig, axes = plt.subplots(5, 4, figsize=(18, 20))
        axes = axes.reshape(5, 4)

        for idx, ((t, p), _) in enumerate(most_confused):
            row = idx // 2
            col = (idx % 2) * 2

            true_name = idx_to_class[t]
            pred_name = idx_to_class[p]

            # Get filepaths for true and predicted classes
            t_imgs = [img_path for img_path, label in test_dataset.data_path if label == t]
            p_imgs = [img_path for img_path, label in test_dataset.data_path if label == p]

            # Sample up to 2 images per class safely
            t_sample = random.sample(t_imgs, min(2, len(t_imgs)))
            p_sample = random.sample(p_imgs, min(2, len(p_imgs)))

            # Fill 2 columns (true vs predicted)
            for i in range(2):
                if i < len(t_sample):
                    img_path = DATASET_PATH / t_sample[i]
                    axes[row, col].imshow(Image.open(img_path).convert("RGB"))
                    axes[row, col].set_title(f"True: {true_name}", fontsize=9)
                    axes[row, col].axis("off")

                if i < len(p_sample):
                    img_path = DATASET_PATH / p_sample[i]
                    axes[row, col + 1].imshow(Image.open(img_path).convert("RGB"))
                    axes[row, col + 1].set_title(f"Pred: {pred_name}", fontsize=9)
                    axes[row, col + 1].axis("off")

        plt.tight_layout()
        sample_img_path = plots_dir / f"{model_name}_most_confused_pairs_samples.png"
        plt.savefig(sample_img_path)
        plt.close()
        print(f"\nSample images of confused pairs saved to: {sample_img_path}")

    # Per-class accuracy CSV
    class_accuracy = []
    for cls in per_class_total:
        acc = per_class_correct[cls] / per_class_total[cls]
        class_accuracy.append(
            (cls, idx_to_class[cls], acc, per_class_correct[cls], per_class_total[cls])
        )

    # Sort high -> low accuracy
    class_accuracy.sort(key=lambda x: x[2], reverse=True)

    csv_path = metric_dir / f"{model_name}_per_class_accuracy.csv"
    with open(csv_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["class_id", "class_name", "accuracy", "correct", "total"])
        for cls, name, acc, correct, total_cls in class_accuracy:
            writer.writerow([cls, name, f"{acc:.4f}", correct, total_cls])

    print(f"\nPer-class accuracy CSV saved to: {csv_path}")

    # Save plots
    save_training_plots(
        model_name=model_name,
        loss_history=checkpoint.get("loss_history", []),
        train_acc_history=checkpoint.get("train_acc_history", []),
        test_acc_history=checkpoint.get("test_acc_history", []),
        epoch_times=checkpoint.get("epoch_times", []),
        output_dir=plots_dir
    )

    print("\nPlots saved")

if __name__ == "__main__":
    param_path = "resnet152_epoch_140.pth"
    inference(param_path)
    summarize_checkpoint_times(param_path)